# NLP Lab Task 5 — N-gram Language Models

Conventions: everything lowercased, sentences wrapped in `<s>` and `</s>`, **V = all words + `</s>`** (no `<s>`),
natural log (`ln`), results rounded to 4 decimals, standard library only.
Very small probabilities are also shown in scientific notation, because rounding them to 4 decimals would just print `0.0`.

In [1]:
import math
from collections import Counter
from itertools import permutations

train = ["the cat sat on the mat", "the cat ate the fish",
         "the dog sat on the rug", "the dog ate the bone"]
test  = ["the cat sat on the rug", "the fish sat on the mat"]

def show(p):
    """Probability as 'rounded (scientific)'."""
    return f"{round(p, 4)}  ({p:.4e})"

## Task 1: Tokenization

In [2]:
def tokenize(sentence):
    """Lowercase, split on spaces, add <s> and </s>."""
    return ["<s>"] + sentence.lower().split() + ["</s>"]

train_tokens = [tokenize(s) for s in train]
test_tokens  = [tokenize(s) for s in test]

print("TRAIN")
for t in train_tokens:
    print(" ", t)
print("TEST")
for t in test_tokens:
    print(" ", t)

total_with    = sum(len(t) for t in train_tokens)
total_without = sum(len(t) - 2 for t in train_tokens)
print(f"\nTotal training tokens WITH markers   : {total_with}")
print(f"Total training tokens WITHOUT markers: {total_without}")

TRAIN
  ['<s>', 'the', 'cat', 'sat', 'on', 'the', 'mat', '</s>']
  ['<s>', 'the', 'cat', 'ate', 'the', 'fish', '</s>']
  ['<s>', 'the', 'dog', 'sat', 'on', 'the', 'rug', '</s>']
  ['<s>', 'the', 'dog', 'ate', 'the', 'bone', '</s>']
TEST
  ['<s>', 'the', 'cat', 'sat', 'on', 'the', 'rug', '</s>']
  ['<s>', 'the', 'fish', 'sat', 'on', 'the', 'mat', '</s>']

Total training tokens WITH markers   : 30
Total training tokens WITHOUT markers: 22


In [3]:
def build_vocab(sentences):
    """All words plus </s>; <s> is NOT part of the vocabulary."""
    vocab = set()
    for s in sentences:
        vocab.update(tokenize(s))
    vocab.discard("<s>")
    return vocab

vocab = build_vocab(train)
V = len(vocab)
print("V =", V)
print("Vocabulary:", sorted(vocab))

V = 11
Vocabulary: ['</s>', 'ate', 'bone', 'cat', 'dog', 'fish', 'mat', 'on', 'rug', 'sat', 'the']


## Task 2: Generating N-grams

In [4]:
def get_ngrams(tokens, n):
    """Return a list of n-gram tuples. Returns [] if n > len(tokens)."""
    if n < 1:
        raise ValueError("n must be >= 1")
    return [tuple(tokens[i:i + n]) for i in range(len(tokens) - n + 1)]

first = train_tokens[0]
print("Sentence :", first)
for n, name in [(1, "Unigrams"), (2, "Bigrams"), (3, "Trigrams")]:
    print(f"{name}:", get_ngrams(first, n))

Sentence : ['<s>', 'the', 'cat', 'sat', 'on', 'the', 'mat', '</s>']
Unigrams: [('<s>',), ('the',), ('cat',), ('sat',), ('on',), ('the',), ('mat',), ('</s>',)]
Bigrams: [('<s>', 'the'), ('the', 'cat'), ('cat', 'sat'), ('sat', 'on'), ('on', 'the'), ('the', 'mat'), ('mat', '</s>')]
Trigrams: [('<s>', 'the', 'cat'), ('the', 'cat', 'sat'), ('cat', 'sat', 'on'), ('sat', 'on', 'the'), ('on', 'the', 'mat'), ('the', 'mat', '</s>')]


In [5]:
# m tokens -> m - n + 1 n-grams, for every sentence and n = 1, 2, 3
for toks in train_tokens + test_tokens:
    m = len(toks)
    for n in (1, 2, 3):
        assert len(get_ngrams(toks, n)) == m - n + 1, (toks, n)
print("Verified: m tokens give m - n + 1 n-grams for n = 1, 2, 3 on all", len(train_tokens + test_tokens), "sentences.")

# n larger than the sentence length -> empty list, no crash
short = ["<s>", "hi", "</s>"]
print("get_ngrams(3 tokens, n=5) ->", get_ngrams(short, 5))
print("get_ngrams([], n=2)       ->", get_ngrams([], 2))

Verified: m tokens give m - n + 1 n-grams for n = 1, 2, 3 on all 6 sentences.
get_ngrams(3 tokens, n=5) -> []
get_ngrams([], n=2)       -> []


## Task 3: Counting N-grams

In [6]:
uni_counts, bi_counts, tri_counts = Counter(), Counter(), Counter()
for toks in train_tokens:
    uni_counts.update(get_ngrams(toks, 1))
    bi_counts.update(get_ngrams(toks, 2))
    tri_counts.update(get_ngrams(toks, 3))

def show_counts(counter, title):
    print(title)
    for ng, c in sorted(counter.items(), key=lambda kv: (-kv[1], kv[0])):
        print(f"  {' '.join(ng):<22}{c}")
    print()

show_counts(uni_counts, "UNIGRAM counts")
show_counts(bi_counts,  "BIGRAM counts")
show_counts(tri_counts, "TRIGRAM counts")

UNIGRAM counts
  the                   8
  </s>                  4
  <s>                   4
  ate                   2
  cat                   2
  dog                   2
  on                    2
  sat                   2
  bone                  1
  fish                  1
  mat                   1
  rug                   1

BIGRAM counts
  <s> the               4
  ate the               2
  on the                2
  sat on                2
  the cat               2
  the dog               2
  bone </s>             1
  cat ate               1
  cat sat               1
  dog ate               1
  dog sat               1
  fish </s>             1
  mat </s>              1
  rug </s>              1
  the bone              1
  the fish              1
  the mat               1
  the rug               1

TRIGRAM counts
  <s> the cat           2
  <s> the dog           2
  sat on the            2
  ate the bone          1
  ate the fish          1
  cat ate the           1
  cat sat on      

In [7]:
top5 = sorted(bi_counts.items(), key=lambda kv: (-kv[1], kv[0]))[:5]
print("5 most common bigrams:")
for ng, c in top5:
    print(f"  {ng}  ->  {c}")

print()
print("Distinct unigrams:", len(uni_counts), "(includes <s> and </s>)")
print("Distinct bigrams :", len(bi_counts))
print("Distinct trigrams:", len(tri_counts))

5 most common bigrams:
  ('<s>', 'the')  ->  4
  ('ate', 'the')  ->  2
  ('on', 'the')  ->  2
  ('sat', 'on')  ->  2
  ('the', 'cat')  ->  2

Distinct unigrams: 12 (includes <s> and </s>)
Distinct bigrams : 18
Distinct trigrams: 19


In [8]:
_tables = {1: uni_counts, 2: bi_counts, 3: tri_counts}

def count_of(ngram):
    """Count of an n-gram (tuple); 0 if unseen (no KeyError, no insertion)."""
    if isinstance(ngram, str):
        ngram = (ngram,)
    ngram = tuple(ngram)
    return _tables.get(len(ngram), {}).get(ngram, 0)

print("count_of(('the','cat'))  [seen]  =", count_of(("the", "cat")))
print("count_of(('cat','dog'))  [unseen]=", count_of(("cat", "dog")))

count_of(('the','cat'))  [seen]  = 2
count_of(('cat','dog'))  [unseen]= 0


## Task 4: Unigram model

`N` = number of predicted tokens = all words + `</s>` (the `<s>` marker is never predicted, so it is not counted).

In [9]:
N_uni = sum(c for (w,), c in uni_counts.items() if w != "<s>")
print("N (words + </s>) =", N_uni)

def unigram_prob(w):
    """MLE: P(w) = C(w) / N"""
    if w == "<s>":
        return 0.0
    return count_of((w,)) / N_uni

print(f"\n{'word':<8}{'C(w)':>6}{'P(w)':>10}")
for w in sorted(vocab):
    print(f"{w:<8}{count_of((w,)):>6}{round(unigram_prob(w), 4):>10}")

total = sum(unigram_prob(w) for w in vocab)
assert math.isclose(total, 1.0)
print("\nSum of unigram probabilities =", total, "-> OK")

N (words + </s>) = 26

word      C(w)      P(w)
</s>         4    0.1538
ate          2    0.0769
bone         1    0.0385
cat          2    0.0769
dog          2    0.0769
fish         1    0.0385
mat          1    0.0385
on           2    0.0769
rug          1    0.0385
sat          2    0.0769
the          8    0.3077

Sum of unigram probabilities = 1.0 -> OK


In [10]:
def unigram_sentence_prob(sentence):
    p = 1.0
    for w in tokenize(sentence)[1:]:      # skip <s>
        p *= unigram_prob(w)
    return p

for s in test:
    print(f"P_unigram('{s}') = {show(unigram_sentence_prob(s))}")

P_unigram('the cat sat on the rug') = 0.0  (2.5499e-07)
P_unigram('the fish sat on the mat') = 0.0  (1.2749e-07)


In [11]:
# Word order does not matter to a unigram model
words = test[0].split()
orderings = set(permutations(words))
probs = {o: unigram_sentence_prob(" ".join(o)) for o in orderings}

same = all(math.isclose(p, probs[tuple(words)], rel_tol=1e-12) for p in probs.values())
print("Distinct word orders of the first test sentence:", len(orderings))
print("All of them have the same unigram probability  :", same)
print()
for o in sorted(orderings)[:3] + [tuple(words)]:
    print(f"  {' '.join(o):<28} -> {probs[o]:.4e}")

Distinct word orders of the first test sentence: 360
All of them have the same unigram probability  : True

  cat on rug sat the the       -> 2.5499e-07
  cat on rug the sat the       -> 2.5499e-07
  cat on rug the the sat       -> 2.5499e-07
  the cat sat on the rug       -> 2.5499e-07


## Task 5: Bigram model (MLE)

In [12]:
def bigram_prob(prev, w):
    """MLE: P(w | prev) = C(prev, w) / C(prev)"""
    c_prev = count_of((prev,))
    if c_prev == 0:
        return 0.0
    return count_of((prev, w)) / c_prev

def next_word_distribution(prev):
    """Dict {next word: probability} for every next word seen after prev."""
    return {w: bigram_prob(prev, w)
            for (v, w) in sorted(bi_counts) if v == prev}

dist_the = next_word_distribution("the")
print("P(. | the):")
for w, p in dist_the.items():
    print(f"  {w:<6}{round(p, 4)}")
assert math.isclose(sum(dist_the.values()), 1.0)
print("Sum =", sum(dist_the.values()), "-> OK")

P(. | the):
  bone  0.125
  cat   0.25
  dog   0.25
  fish  0.125
  mat   0.125
  rug   0.125
Sum = 1.0 -> OK


In [13]:
def bigram_sentence_prob(sentence):
    """Chain rule: product of P(w_i | w_{i-1}) including <s> -> w1 and w_n -> </s>."""
    p = 1.0
    for prev, w in get_ngrams(tokenize(sentence), 2):
        p *= bigram_prob(prev, w)
    return p

for s in test:
    print(f"P_bigram('{s}') = {show(bigram_sentence_prob(s))}")

P_bigram('the cat sat on the rug') = 0.0156  (1.5625e-02)
P_bigram('the fish sat on the mat') = 0.0  (0.0000e+00)


In [14]:
def predict_next(prev):
    """Most probable next word; ties broken alphabetically."""
    dist = next_word_distribution(prev)
    if not dist:
        return None
    return sorted(dist.items(), key=lambda kv: (-kv[1], kv[0]))[0][0]

for w in ["<s>", "the", "cat", "sat", "on", "dog", "ate", "fish"]:
    print(f"predict_next({w!r:<8}) -> {predict_next(w)!r}   (dist: { {k: round(v, 4) for k, v in next_word_distribution(w).items()} })")

predict_next('<s>'   ) -> 'the'   (dist: {'the': 1.0})
predict_next('the'   ) -> 'cat'   (dist: {'bone': 0.125, 'cat': 0.25, 'dog': 0.25, 'fish': 0.125, 'mat': 0.125, 'rug': 0.125})
predict_next('cat'   ) -> 'ate'   (dist: {'ate': 0.5, 'sat': 0.5})
predict_next('sat'   ) -> 'on'   (dist: {'on': 1.0})
predict_next('on'    ) -> 'the'   (dist: {'the': 1.0})
predict_next('dog'   ) -> 'ate'   (dist: {'ate': 0.5, 'sat': 0.5})
predict_next('ate'   ) -> 'the'   (dist: {'the': 1.0})
predict_next('fish'  ) -> '</s>'   (dist: {'</s>': 1.0})


## Task 6: Trigram model

The first word is predicted with the bigram model `P(w1 | <s>)`; every later word uses the trigram model.

In [15]:
def trigram_prob(w1, w2, w3):
    """MLE: P(w3 | w1, w2) = C(w1, w2, w3) / C(w1, w2)"""
    c_ctx = count_of((w1, w2))
    if c_ctx == 0:
        return 0.0
    return count_of((w1, w2, w3)) / c_ctx

def trigram_sentence_prob(sentence):
    toks = tokenize(sentence)
    p = bigram_prob("<s>", toks[1])                 # first word: bigram model
    for i in range(2, len(toks)):                    # remaining words: trigram model
        p *= trigram_prob(toks[i - 2], toks[i - 1], toks[i])
    return p

for s in test:
    print(f"P_trigram('{s}') = {show(trigram_sentence_prob(s))}")

P_trigram('the cat sat on the rug') = 0.125  (1.2500e-01)
P_trigram('the fish sat on the mat') = 0.0  (0.0000e+00)


In [16]:
print(f"{'N-gram':<10}{'distinct types':>15}")
print(f"{'unigram':<10}{len(uni_counts):>15}")
print(f"{'bigram':<10}{len(bi_counts):>15}")
print(f"{'trigram':<10}{len(tri_counts):>15}")

N-gram     distinct types
unigram                12
bigram                 18
trigram                19


In [17]:
for s in test:
    toks = tokenize(s)
    ub = [g for g in get_ngrams(toks, 2) if count_of(g) == 0]
    ut = [g for g in get_ngrams(toks, 3) if count_of(g) == 0]
    print(f"Sentence: '{s}'")
    print("  Unseen bigrams :", ub)
    print("  Unseen trigrams:", ut)
    if len(ut) > len(ub):
        verdict = "TRIGRAM model"
    elif len(ub) > len(ut):
        verdict = "BIGRAM model"
    else:
        verdict = "neither (equal)"
    print(f"  {len(ub)} unseen bigrams vs {len(ut)} unseen trigrams -> more unseen N-grams: {verdict}\n")

Sentence: 'the cat sat on the rug'
  Unseen bigrams : []
  Unseen trigrams: []
  0 unseen bigrams vs 0 unseen trigrams -> more unseen N-grams: neither (equal)

Sentence: 'the fish sat on the mat'
  Unseen bigrams : [('fish', 'sat')]
  Unseen trigrams: [('<s>', 'the', 'fish'), ('the', 'fish', 'sat'), ('fish', 'sat', 'on')]
  1 unseen bigrams vs 3 unseen trigrams -> more unseen N-grams: TRIGRAM model



## Task 7: Zero-probability problem

In [18]:
def unseen_bigrams(sentence):
    """Every bigram of the sentence that never occurs in training."""
    return [g for g in get_ngrams(tokenize(sentence), 2) if count_of(g) == 0]

for s in test:
    print(f"Sentence: '{s}'")
    print("  Unseen bigrams:", unseen_bigrams(s))
    for g in get_ngrams(tokenize(s), 2):
        print(f"    P({g[1]} | {g[0]}) = {round(bigram_prob(*g), 4)}")
    print("  Sentence probability (bigram MLE):", show(bigram_sentence_prob(s)))
    print()

assert any(bigram_sentence_prob(s) == 0 for s in test)
print("Confirmed: at least one test sentence gets probability 0.")

Sentence: 'the cat sat on the rug'
  Unseen bigrams: []
    P(the | <s>) = 1.0
    P(cat | the) = 0.25
    P(sat | cat) = 0.5
    P(on | sat) = 1.0
    P(the | on) = 1.0
    P(rug | the) = 0.125
    P(</s> | rug) = 1.0
  Sentence probability (bigram MLE): 0.0156  (1.5625e-02)

Sentence: 'the fish sat on the mat'
  Unseen bigrams: [('fish', 'sat')]
    P(the | <s>) = 1.0
    P(fish | the) = 0.125
    P(sat | fish) = 0.0
    P(on | sat) = 1.0
    P(the | on) = 1.0
    P(mat | the) = 0.125
    P(</s> | mat) = 1.0
  Sentence probability (bigram MLE): 0.0  (0.0000e+00)

Confirmed: at least one test sentence gets probability 0.


In [19]:
words_only = sorted(w for w in vocab if w != "</s>")
possible = len(words_only) ** 2
seen_pairs = sum(1 for (a, b) in bi_counts if a in words_only and b in words_only)
unseen_pairs = possible - seen_pairs

print("Words (markers excluded):", len(words_only))
print("Possible word pairs     :", possible)
print("Seen in training        :", seen_pairs)
print("Never seen              :", unseen_pairs)
print(f"Percentage never seen   : {round(100 * unseen_pairs / possible, 4)}%")

Words (markers excluded): 10
Possible word pairs     : 100
Seen in training        : 13
Never seen              : 87
Percentage never seen   : 87.0%


In [20]:
def safe_log(p):
    """Natural log that returns -inf for p = 0 instead of raising ValueError."""
    return float("-inf") if p == 0 else math.log(p)

lp = sum(safe_log(bigram_prob(a, b)) for a, b in get_ngrams(tokenize(test[1]), 2))
print(f"log P_bigram('{test[1]}') = {lp}")

log P_bigram('the fish sat on the mat') = -inf


## Task 8: Smoothing

In [21]:
def bigram_prob_addone(prev, w):
    """Add-one: (C(prev, w) + 1) / (C(prev) + V)"""
    return (count_of((prev, w)) + 1) / (count_of((prev,)) + V)

print(f"{'':<16}{'MLE':>10}{'Add-one':>10}")
for prev, w in [("fish", "sat"), ("the", "cat")]:
    print(f"P({w} | {prev})".ljust(16) + f"{round(bigram_prob(prev, w), 4):>10}{round(bigram_prob_addone(prev, w), 4):>10}")

                       MLE   Add-one
P(sat | fish)          0.0    0.0833
P(cat | the)          0.25    0.1579


In [22]:
# Every possible previous word = <s> and every real word. </s> is excluded: it ends a sentence, so nothing can follow it.
prev_words = ["<s>"] + sorted(w for w in vocab if w != "</s>")
for prev in prev_words:
    total = sum(bigram_prob_addone(prev, w) for w in vocab)
    assert math.isclose(total, 1.0), (prev, total)
print("Add-one smoothed next-word distributions (over all V words) sum to 1 for all", len(prev_words), "possible previous words -> OK")

Add-one smoothed next-word distributions (over all V words) sum to 1 for all 11 possible previous words -> OK


In [23]:
def bigram_prob_addk(prev, w, k):
    """Add-k: (C(prev, w) + k) / (C(prev) + k*V)"""
    return (count_of((prev, w)) + k) / (count_of((prev,)) + k * V)

KS = [1, 0.5, 0.1, 0.01]
print(f"{'k':<8}{'P(sat | fish)':>15}")
for k in KS:
    print(f"{k:<8}{round(bigram_prob_addk('fish', 'sat', k), 4):>15}")

k         P(sat | fish)
1                0.0833
0.5              0.0769
0.1              0.0476
0.01              0.009


## Task 9: Perplexity and model comparison

`prob_fn(prev, w)` returns the probability of word `w` given the previous token, so unigram and bigram models share one interface.
Perplexity = exp( − (1/N) · Σ ln P ), where N counts every predicted token (words + `</s>`) across all sentences.

In [24]:
def perplexity(sentences, prob_fn):
    log_sum, n = 0.0, 0
    for s in sentences:
        toks = tokenize(s)
        for i in range(1, len(toks)):
            p = prob_fn(toks[i - 1], toks[i])
            if p == 0:
                return float("inf")
            log_sum += math.log(p)
            n += 1
    return math.exp(-log_sum / n)

def unigram_prob_addone(prev, w):
    """P(w) = (C(w) + 1) / (N + V)   (prev is ignored)"""
    return (count_of((w,)) + 1) / (N_uni + V)

results = {
    "Bigram MLE":        perplexity(test, bigram_prob),
    "Bigram add-one":    perplexity(test, bigram_prob_addone),
    "Unigram add-one":   perplexity(test, unigram_prob_addone),
}
print(f"{'Model':<20}{'Perplexity':>12}")
for name, pp in results.items():
    print(f"{name:<20}{(round(pp, 4) if pp != float('inf') else 'inf'):>12}")

Model                 Perplexity
Bigram MLE                   inf
Bigram add-one            5.7969
Unigram add-one            9.137


In [25]:
pp_k = {k: perplexity(test, lambda p, w, k=k: bigram_prob_addk(p, w, k)) for k in KS}
print(f"{'k':<8}{'Perplexity':>12}")
for k, pp in pp_k.items():
    print(f"{k:<8}{round(pp, 4):>12}")
best_k_full = min(pp_k, key=pp_k.get)
print("\nBest k (full training set, scored on test):", best_k_full)

k         Perplexity
1             5.7969
0.5           4.5703
0.1           2.9559
0.01          2.6272

Best k (full training set, scored on test): 0.01


In [26]:
# Train on the first 3 sentences only, V still built from all 4. Validation = 4th training sentence.
def build_counts(sentences):
    u, b = Counter(), Counter()
    for s in sentences:
        t = tokenize(s)
        u.update(get_ngrams(t, 1))
        b.update(get_ngrams(t, 2))
    return u, b

def make_bigram_addk(sentences, V_size):
    u, b = build_counts(sentences)
    def prob(prev, w, k):
        return (b.get((prev, w), 0) + k) / (u.get((prev,), 0) + k * V_size)
    return prob

train3, valid = train[:3], train[3:]
model3 = make_bigram_addk(train3, V)       # V = 11, from all four sentences

pp_valid = {k: perplexity(valid, lambda p, w, k=k: model3(p, w, k)) for k in KS}
pp_test3 = {k: perplexity(test,  lambda p, w, k=k: model3(p, w, k)) for k in KS}

print(f"{'k':<8}{'Validation PP':>15}{'Test PP':>12}")
for k in KS:
    print(f"{k:<8}{round(pp_valid[k], 4):>15}{round(pp_test3[k], 4):>12}")

best_k_valid = min(pp_valid, key=pp_valid.get)
best_k_test  = min(pp_test3, key=pp_test3.get)

print()
print(f"Best k chosen on validation data : {best_k_valid}")
print(f"Test perplexity with that k      : {round(pp_test3[best_k_valid], 4)}")
print(f"Best k chosen directly on test   : {best_k_test}")
print(f"Test perplexity with that k      : {round(pp_test3[best_k_test], 4)}")
if best_k_valid == best_k_test:
    print("-> Validation picked the same k as the test set, so nothing was lost by not touching the test data.")
else:
    gap = pp_test3[best_k_valid] - pp_test3[best_k_test]
    print(f"-> Validation picked a different k; it costs {round(gap, 4)} perplexity compared with the test-optimal k.")

k         Validation PP     Test PP
1                8.5857      5.7406
0.5              8.0163      4.4539
0.1               8.026      2.7778
0.01            13.1304      2.4258

Best k chosen on validation data : 0.5
Test perplexity with that k      : 4.4539
Best k chosen directly on test   : 0.01
Test perplexity with that k      : 2.4258
-> Validation picked a different k; it costs 2.028 perplexity compared with the test-optimal k.
